# 04D1F_v2 — Archived LANDFIRE Inventory & Provenance Validation

**Build ID: `04D1F_V2_LANDFIRE_ARCHIVE_INVENTORY_PROVENANCE`**

This notebook is **inventory-only by default**.

It does **not** download the large LANDFIRE archives.

The goal is to turn the two official sources supplied by the LANDFIRE Help
Desk into a machine-readable acquisition plan for the two remaining frozen
vintages:

| Fire year | Frozen LANDFIRE vintage | Production role |
|---|---|---|
| 2021 | LF2019 Limited (`LF2019L`) | 2021-capable retrospective landscape |
| 2022 | LF2020 (`LF2020`, v2.2.0 / 220) | 2022-capable retrospective landscape |

Required products:

- FBFM40
- CC
- CH
- CBH
- CBD

Official sources:

- LF2020 ScienceBase DOI: `https://doi.org/10.5066/P1LJEZ3G`
- LF2019 Limited Help Desk directory:
  `https://edcftp.cr.usgs.gov/project/landfire/for_U_of_Cali_SD/`

## What this notebook does

1. resolves the LF2020 DOI to its ScienceBase item;
2. traverses the ScienceBase release hierarchy through the public JSON API;
3. inventories attached files and published checksum/size metadata;
4. crawls the Help Desk FTP/HTTP directory listing for LF2019 Limited;
5. identifies candidate ZIPs for the five required products;
6. ranks candidates using product, version and CONUS naming evidence;
7. writes a provenance/source manifest and acquisition plan;
8. **hard-stops before downloading archives**.

No frozen feature architecture or vintage mapping is changed here.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1F_V2_LANDFIRE_ARCHIVE_INVENTORY_PROVENANCE"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_inventory_v2"
)

RAW_ARCHIVE_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" /
    "archive_acquisition"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RAW_ARCHIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LF2020_DOI_URL = (
    "https://doi.org/10.5066/P1LJEZ3G"
)

LF2019L_HELPDESK_URL = (
    "https://edcftp.cr.usgs.gov/"
    "project/landfire/for_U_of_Cali_SD/"
)

REQUIRED_PRODUCTS = [
    "FBFM40",
    "CC",
    "CH",
    "CBH",
    "CBD",
]

SOURCE_PLAN = {
    "LF2019L": {
        "fire_year": 2021,
        "source_kind": "USGS_helpdesk_directory",
        "source_url": LF2019L_HELPDESK_URL,
        "expected_version_suffix": "210",
        "expected_capable_year": 2021,
    },
    "LF2020": {
        "fire_year": 2022,
        "source_kind": "USGS_ScienceBase",
        "source_url": LF2020_DOI_URL,
        "expected_version_suffix": "220",
        "expected_capable_year": 2022,
    },
}

# ------------------------------------------------------------------
# SAFETY:
# This notebook may inspect URLs and issue HEAD/metadata requests,
# but it will not download archive bodies.
# ------------------------------------------------------------------
ALLOW_ARCHIVE_DOWNLOAD = False

REQUEST_TIMEOUT_S = 90
MAX_SCIENCEBASE_ITEMS = 5000
SCIENCEBASE_HYDRATE_SLEEP_S = 0.03
FTP_MAX_DEPTH = 3
FTP_MAX_PAGES = 100

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("ALLOW_ARCHIVE_DOWNLOAD:", ALLOW_ARCHIVE_DOWNLOAD)

assert ALLOW_ARCHIVE_DOWNLOAD is False

## V2 correction — hydrate ScienceBase search results

ScienceBase `/catalog/items` queries return search-result item summaries.
Those summaries are sufficient for hierarchy discovery but may omit attached
`files`.

V1 incorrectly flattened those summaries directly, so the inventory could
report zero files even when descendant items had attachments.

V2 now:

1. discovers descendants with the official `ancestors` filter;
2. fetches the **full JSON for every descendant item ID**;
3. inventories `files` only from those hydrated full items;
4. records hydration failures explicitly;
5. returns an empty DataFrame with the expected schema rather than crashing
   with `KeyError` if no file attachment is present.

No archive is downloaded.


## 1. Dependencies and HTTP session

In [ ]:
required = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "numpy": "numpy",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
from urllib.parse import (
    urljoin,
    urlparse,
    unquote,
)
import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option(
    "display.max_columns",
    200,
)
pd.set_option(
    "display.max_rows",
    500,
)
pd.set_option(
    "display.width",
    360,
)

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "UCSD-wildfire-fireline-effectiveness/"
        "04D1F-LANDFIRE-inventory"
    )
})

## 2. Frozen acquisition contract

In [ ]:
contract_rows = []

for version_key, spec in SOURCE_PLAN.items():
    for product in REQUIRED_PRODUCTS:
        contract_rows.append({
            "version_key": version_key,
            "fire_year": spec["fire_year"],
            "product": product,
            "source_kind": spec["source_kind"],
            "official_source_url": spec["source_url"],
            "expected_version_suffix": (
                spec["expected_version_suffix"]
            ),
            "expected_capable_year": (
                spec["expected_capable_year"]
            ),
            "download_required": True,
        })

acquisition_contract = pd.DataFrame(
    contract_rows
)

display(
    acquisition_contract
)

acquisition_contract.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_ACQUISITION_CONTRACT_V2.csv",
    index=False,
)

assert len(
    acquisition_contract
) == 10

## 3. Product-name recognition

LANDFIRE filenames have changed across releases. Candidate identification
therefore uses token-aware aliases rather than assuming a filename before the
official inventory is inspected.

Version suffixes (`210`, `220`) are used as **ranking evidence**, not as proof.

In [ ]:
PRODUCT_ALIASES = {
    "FBFM40": {
        "FBFM40",
        "F40",
        "FUELMODEL40",
    },
    "CC": {
        "CC",
        "CANOPYCOVER",
    },
    "CH": {
        "CH",
        "CANOPYHEIGHT",
    },
    "CBH": {
        "CBH",
        "CANOPYBASEHEIGHT",
    },
    "CBD": {
        "CBD",
        "CANOPYBULKDENSITY",
    },
}

CONUS_ALIASES = {
    "CONUS",
    "L48",
    "LOWER48",
}

ARCHIVE_EXTENSIONS = {
    ".zip",
    ".7z",
    ".tar",
    ".gz",
    ".tgz",
}


def normalized_name_tokens(name):
    text = unquote(
        str(name)
    ).upper()

    tokens = [
        t
        for t in re.split(
            r"[^A-Z0-9]+",
            text,
        )
        if t
    ]

    compact = re.sub(
        r"[^A-Z0-9]",
        "",
        text,
    )

    return set(tokens), compact


def detect_product(name):
    tokens, compact = normalized_name_tokens(
        name
    )

    matches = []

    # Check longest / most specific products first.
    for product in [
        "FBFM40",
        "CBH",
        "CBD",
        "CC",
        "CH",
    ]:
        aliases = PRODUCT_ALIASES[
            product
        ]

        hit = False

        for alias in aliases:
            alias_compact = re.sub(
                r"[^A-Z0-9]",
                "",
                alias.upper(),
            )

            # Short aliases must be exact tokens to avoid false matches.
            if len(alias_compact) <= 3:
                if alias_compact in tokens:
                    hit = True
            else:
                if (
                    alias_compact in tokens
                    or alias_compact in compact
                ):
                    hit = True

            if hit:
                break

        if hit:
            matches.append(
                product
            )

    if len(matches) == 1:
        return matches[0]

    return None


def is_archive_name(name):
    lower = str(name).lower()

    return (
        lower.endswith(".zip")
        or lower.endswith(".7z")
        or lower.endswith(".tar")
        or lower.endswith(".tar.gz")
        or lower.endswith(".tgz")
    )


# Synthetic recognition QA.
tests = {
    "LC21_F40_210.zip": "FBFM40",
    "LC21_CC_210.zip": "CC",
    "LC21_CH_210.zip": "CH",
    "LC21_CBH_210.zip": "CBH",
    "LC21_CBD_210.zip": "CBD",
    "LF2020_FBFM40_CONUS.zip": "FBFM40",
}

for filename, expected in tests.items():
    assert detect_product(
        filename
    ) == expected, filename

print(
    "PRODUCT-NAME RECOGNITION: PASS"
)

## 4. Resolve LF2020 DOI and query ScienceBase

ScienceBase's public item API exposes item metadata as JSON by appending
`?format=json`.

The release may have nested child items, so this stage inventories the root item
plus all descendants rather than assuming files live on the DOI landing item.

In [ ]:
def resolve_doi_to_sciencebase(
    doi_url,
):
    r = session.get(
        doi_url,
        allow_redirects=True,
        timeout=REQUEST_TIMEOUT_S,
    )
    r.raise_for_status()

    final_url = r.url

    m = re.search(
        r"/catalog/item/([A-Za-z0-9]+)",
        final_url,
    )

    if not m:
        raise RuntimeError(
            "DOI did not resolve to a ScienceBase item URL. "
            f"Final URL: {final_url}"
        )

    return {
        "doi_url": doi_url,
        "final_url": final_url,
        "sciencebase_item_id": (
            m.group(1)
        ),
    }


lf2020_resolution = (
    resolve_doi_to_sciencebase(
        LF2020_DOI_URL
    )
)

display(
    pd.DataFrame(
        [lf2020_resolution]
    )
)

(
    OUTPUT_ROOT /
    "LF2020_DOI_RESOLUTION_V2.json"
).write_text(
    json.dumps(
        lf2020_resolution,
        indent=2,
    ),
    encoding="utf-8",
)

In [ ]:

SCIENCEBASE_ROOT = (
    "https://www.sciencebase.gov"
)


def sciencebase_item_json(
    item_id,
):
    """
    Fetch the complete sbJSON representation of one ScienceBase item.
    Full item JSON is required for attached `files`.
    """
    url = (
        f"{SCIENCEBASE_ROOT}/catalog/"
        f"item/{item_id}"
    )

    r = session.get(
        url,
        params={
            "format": "json",
        },
        timeout=REQUEST_TIMEOUT_S,
    )

    r.raise_for_status()

    payload = r.json()

    if not isinstance(
        payload,
        dict,
    ):
        raise RuntimeError(
            f"ScienceBase item {item_id} did not return an object."
        )

    if str(
        payload.get(
            "id",
            item_id,
        )
    ) != str(
        item_id
    ):
        raise RuntimeError(
            f"ScienceBase item-ID mismatch for {item_id}."
        )

    return payload


def sciencebase_search(
    filters,
):
    """
    Paged ScienceBase search returning SEARCH SUMMARIES.

    These summaries are for discovery only; callers must hydrate item IDs
    before relying on attachment-level fields such as `files`.
    """
    rows = []
    offset = 0
    page_size = 1000

    while True:
        params = [
            ("q", ""),
            ("format", "json"),
            ("max", str(page_size)),
            ("offset", str(offset)),
        ]

        for filter_value in filters:
            params.append(
                (
                    "filter",
                    filter_value,
                )
            )

        r = session.get(
            f"{SCIENCEBASE_ROOT}/catalog/items",
            params=params,
            timeout=REQUEST_TIMEOUT_S,
        )

        r.raise_for_status()

        payload = r.json()

        items = payload.get(
            "items",
            [],
        )

        if not isinstance(
            items,
            list,
        ):
            raise RuntimeError(
                "ScienceBase search did not return an items list."
            )

        rows.extend(
            items
        )

        if len(
            rows
        ) > MAX_SCIENCEBASE_ITEMS:
            raise RuntimeError(
                "ScienceBase search exceeded safety item limit."
            )

        if len(
            items
        ) < page_size:
            break

        offset += page_size

    return rows


def sciencebase_descendant_summaries(
    root_item_id,
):
    """
    Official ScienceBase `ancestors` filter:
    all descendants at any depth.
    """
    return sciencebase_search(
        [
            f"ancestors={root_item_id}",
        ]
    )


def hydrate_sciencebase_items(
    summaries,
):
    """
    Convert lightweight search-result summaries into full item JSON.

    Returns:
      hydrated_items, hydration_log
    """
    hydrated = []
    log_rows = []

    seen = set()

    for i, summary in enumerate(
        summaries,
        start=1,
    ):
        item_id = summary.get(
            "id"
        )

        if not item_id:
            log_rows.append({
                "item_id": None,
                "summary_title": summary.get(
                    "title"
                ),
                "hydration_status": "missing_item_id",
                "error": None,
            })
            continue

        if item_id in seen:
            continue

        seen.add(
            item_id
        )

        try:
            item = sciencebase_item_json(
                item_id
            )

            hydrated.append(
                item
            )

            log_rows.append({
                "item_id": item_id,
                "summary_title": summary.get(
                    "title"
                ),
                "hydration_status": "success",
                "has_children": item.get(
                    "hasChildren"
                ),
                "file_count": len(
                    item.get(
                        "files",
                        [],
                    )
                    or []
                ),
                "error": None,
            })

        except Exception as exc:
            log_rows.append({
                "item_id": item_id,
                "summary_title": summary.get(
                    "title"
                ),
                "hydration_status": "failed",
                "has_children": None,
                "file_count": None,
                "error": (
                    f"{type(exc).__name__}: "
                    f"{str(exc)[:1000]}"
                ),
            })

        if (
            SCIENCEBASE_HYDRATE_SLEEP_S
            > 0
        ):
            time.sleep(
                SCIENCEBASE_HYDRATE_SLEEP_S
            )

    return (
        hydrated,
        pd.DataFrame(
            log_rows
        ),
    )


root_id = lf2020_resolution[
    "sciencebase_item_id"
]

root_item = sciencebase_item_json(
    root_id
)

descendant_summaries = (
    sciencebase_descendant_summaries(
        root_id
    )
)

(
    hydrated_descendants,
    sciencebase_hydration_log,
) = hydrate_sciencebase_items(
    descendant_summaries
)

print(
    "ScienceBase root title:",
    root_item.get(
        "title"
    ),
)

print(
    "Root hasChildren:",
    root_item.get(
        "hasChildren"
    ),
)

print(
    "Descendant search summaries:",
    f"{len(descendant_summaries):,}",
)

print(
    "Hydrated descendant items:",
    f"{len(hydrated_descendants):,}",
)

print(
    "Hydration failures:",
    f"{int((sciencebase_hydration_log['hydration_status'] == 'failed').sum()) if len(sciencebase_hydration_log) else 0:,}",
)

print(
    "Hydrated descendant items with attached files:",
    f"{sum(bool(item.get('files')) for item in hydrated_descendants):,}",
)

assert (
    "LANDFIRE"
    in str(
        root_item.get(
            "title",
            ""
        )
    ).upper()
)

sciencebase_hydration_log.to_csv(
    OUTPUT_ROOT /
    "LF2020_SCIENCEBASE_HYDRATION_LOG_V2.csv",
    index=False,
)


## 5. Flatten ScienceBase files and checksum metadata

In [ ]:

SCIENCEBASE_FILE_COLUMNS = [
    "source",
    "item_id",
    "item_title",
    "item_parent_id",
    "item_has_children",
    "file_name",
    "file_url",
    "file_size_bytes",
    "content_type",
    "checksum_type",
    "checksum_value",
    "product_detected",
    "is_archive",
    "raw_file_json",
]


def flatten_sciencebase_files(
    root_item,
    hydrated_descendants,
):
    """
    Flatten attachments from FULL ScienceBase item JSON only.

    The root item is already full JSON. Every descendant passed here has been
    individually hydrated through `/catalog/item/<id>?format=json`.
    """
    all_items = [
        root_item,
        *hydrated_descendants,
    ]

    rows = []

    for item in all_items:

        item_id = item.get(
            "id"
        )

        item_title = item.get(
            "title"
        )

        files = (
            item.get(
                "files",
                [],
            )
            or []
        )

        for f in files:

            if not isinstance(
                f,
                dict,
            ):
                continue

            name = (
                f.get(
                    "name"
                )
                or f.get(
                    "title"
                )
                or ""
            )

            url = (
                f.get(
                    "url"
                )
                or f.get(
                    "downloadUri"
                )
                or f.get(
                    "downloadURL"
                )
            )

            checksum = (
                f.get(
                    "checksum"
                )
                or f.get(
                    "md5"
                )
                or f.get(
                    "sha1"
                )
            )

            checksum_type = None
            checksum_value = None

            if isinstance(
                checksum,
                dict,
            ):
                checksum_type = (
                    checksum.get(
                        "type"
                    )
                    or checksum.get(
                        "algorithm"
                    )
                )

                checksum_value = (
                    checksum.get(
                        "value"
                    )
                    or checksum.get(
                        "checksum"
                    )
                )

            elif checksum is not None:
                checksum_value = str(
                    checksum
                )

            rows.append({
                "source": (
                    "LF2020_ScienceBase"
                ),
                "item_id": item_id,
                "item_title": item_title,
                "item_parent_id": item.get(
                    "parentId"
                ),
                "item_has_children": item.get(
                    "hasChildren"
                ),
                "file_name": name,
                "file_url": url,
                "file_size_bytes": (
                    f.get(
                        "size"
                    )
                    or f.get(
                        "sizeBytes"
                    )
                ),
                "content_type": (
                    f.get(
                        "contentType"
                    )
                ),
                "checksum_type": checksum_type,
                "checksum_value": checksum_value,
                "product_detected": (
                    detect_product(
                        name
                    )
                ),
                "is_archive": (
                    is_archive_name(
                        name
                    )
                ),
                "raw_file_json": (
                    json.dumps(
                        f,
                        default=str,
                        sort_keys=True,
                    )
                ),
            })

    # Schema-stable even when no attachment is found.
    return pd.DataFrame(
        rows,
        columns=SCIENCEBASE_FILE_COLUMNS,
    )


sciencebase_files = (
    flatten_sciencebase_files(
        root_item,
        hydrated_descendants,
    )
)

print(
    "ScienceBase attached files:",
    f"{len(sciencebase_files):,}",
)

print(
    "ScienceBase items represented by attached files:",
    f"{sciencebase_files['item_id'].nunique() if len(sciencebase_files) else 0:,}",
)

detected_files = (
    sciencebase_files[
        sciencebase_files[
            "product_detected"
        ].notna()
    ].copy()
)

print(
    "Attached files recognized as required products:",
    f"{len(detected_files):,}",
)

display(
    detected_files[
        [
            "item_title",
            "file_name",
            "file_size_bytes",
            "checksum_type",
            "checksum_value",
            "product_detected",
            "is_archive",
        ]
    ].head(
        300
    )
)

sciencebase_files.to_csv(
    OUTPUT_ROOT /
    "LF2020_SCIENCEBASE_FILE_INVENTORY_V2.csv",
    index=False,
)

# This is diagnostic, not yet a hard requirement. Candidate selection later
# determines whether all five requested product archives are discoverable.
if len(
    sciencebase_files
) == 0:
    print(
        "WARNING: no ScienceBase attachments found after full-item hydration. "
        "The notebook will continue safely and report inventory readiness as false."
    )


## 6. Crawl the LF2019 Limited Help Desk directory

The supplied `edcftp.cr.usgs.gov` path is an HTTP-accessible directory listing.
The crawler stays strictly underneath the supplied Help Desk root and uses a
small depth/page limit.

It records links and listing text only; archive bodies are not downloaded.

In [ ]:
def same_url_tree(
    root_url,
    candidate_url,
):
    root = urlparse(
        root_url
    )

    candidate = urlparse(
        candidate_url
    )

    return bool(
        candidate.scheme
        in {
            "http",
            "https",
        }
        and candidate.netloc
        == root.netloc
        and candidate.path.startswith(
            root.path
        )
    )


def parse_human_size(
    text,
):
    if text is None:
        return np.nan

    s = str(text).strip()

    m = re.fullmatch(
        r"([0-9.]+)\s*([KMGT]?)",
        s,
        re.I,
    )

    if not m:
        return np.nan

    value = float(
        m.group(1)
    )

    unit = m.group(2).upper()

    mult = {
        "": 1,
        "K": 1024,
        "M": 1024 ** 2,
        "G": 1024 ** 3,
        "T": 1024 ** 4,
    }[
        unit
    ]

    return int(
        value * mult
    )


def crawl_directory_listing(
    root_url,
    max_depth=FTP_MAX_DEPTH,
    max_pages=FTP_MAX_PAGES,
):
    queue = [
        (
            root_url,
            0,
        )
    ]

    visited = set()
    rows = []

    while queue:
        page_url, depth = queue.pop(
            0
        )

        if page_url in visited:
            continue

        if len(
            visited
        ) >= max_pages:
            raise RuntimeError(
                "FTP directory crawl exceeded page safety limit."
            )

        visited.add(
            page_url
        )

        r = session.get(
            page_url,
            timeout=REQUEST_TIMEOUT_S,
        )

        r.raise_for_status()

        soup = BeautifulSoup(
            r.text,
            "html.parser",
        )

        for a in soup.find_all(
            "a",
            href=True,
        ):
            href = a.get(
                "href"
            )

            label = a.get_text(
                " ",
                strip=True,
            )

            if not href:
                continue

            if href in {
                "../",
                "/",
            }:
                continue

            absolute = urljoin(
                page_url,
                href,
            )

            if not same_url_tree(
                root_url,
                absolute,
            ):
                continue

            path = urlparse(
                absolute
            ).path

            is_dir = (
                href.endswith("/")
                or path.endswith("/")
            )

            # Apache directory indexes usually expose modified/size in the
            # surrounding text. Preserve it for human inspection, but do not
            # make selection depend on this fragile presentation field.
            parent_text = (
                a.parent.get_text(
                    " ",
                    strip=True,
                )
                if a.parent
                else ""
            )

            rows.append({
                "source": "LF2019L_helpdesk",
                "directory_url": page_url,
                "depth": depth,
                "name": (
                    label
                    or Path(
                        path
                    ).name
                ),
                "url": absolute,
                "is_directory": is_dir,
                "listing_context": parent_text,
                "product_detected": (
                    None
                    if is_dir
                    else detect_product(
                        label
                        or Path(
                            path
                        ).name
                    )
                ),
                "is_archive": (
                    False
                    if is_dir
                    else is_archive_name(
                        label
                        or Path(
                            path
                        ).name
                    )
                ),
            })

            if (
                is_dir
                and depth < max_depth
                and absolute not in visited
            ):
                queue.append(
                    (
                        absolute,
                        depth + 1,
                    )
                )

    return pd.DataFrame(
        rows
    )


lf2019l_inventory = (
    crawl_directory_listing(
        LF2019L_HELPDESK_URL
    )
)

print(
    "Help Desk directory entries:",
    f"{len(lf2019l_inventory):,}",
)

display(
    lf2019l_inventory[
        (
            ~lf2019l_inventory[
                "is_directory"
            ]
        )
        & (
            lf2019l_inventory[
                "product_detected"
            ].notna()
        )
    ][
        [
            "depth",
            "name",
            "product_detected",
            "is_archive",
            "url",
        ]
    ].head(200)
)

lf2019l_inventory.to_csv(
    OUTPUT_ROOT /
    "LF2019L_HELPDESK_DIRECTORY_INVENTORY_V2.csv",
    index=False,
)

## 7. Optional HEAD metadata

HEAD requests do not download archive bodies. They are used only to collect
server-reported content length/type when available.

Servers that do not support HEAD are recorded rather than treated as failures.

In [ ]:
def head_metadata(
    url,
):
    try:
        r = session.head(
            url,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT_S,
        )

        return {
            "head_status": r.status_code,
            "head_final_url": r.url,
            "head_content_length": (
                pd.to_numeric(
                    r.headers.get(
                        "Content-Length"
                    ),
                    errors="coerce",
                )
            ),
            "head_content_type": (
                r.headers.get(
                    "Content-Type"
                )
            ),
            "head_etag": (
                r.headers.get(
                    "ETag"
                )
            ),
            "head_last_modified": (
                r.headers.get(
                    "Last-Modified"
                )
            ),
        }

    except Exception as exc:
        return {
            "head_status": np.nan,
            "head_final_url": None,
            "head_content_length": np.nan,
            "head_content_type": None,
            "head_etag": None,
            "head_last_modified": None,
            "head_error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:500]}"
            ),
        }

## 8. Build candidate table for the 10 required archives

In [ ]:
def candidate_score(
    version_key,
    product,
    name,
    is_archive,
):
    tokens, compact = normalized_name_tokens(
        name
    )

    expected_suffix = (
        SOURCE_PLAN[
            version_key
        ][
            "expected_version_suffix"
        ]
    )

    score = 0
    reasons = []

    detected = detect_product(
        name
    )

    if detected == product:
        score += 100
        reasons.append(
            "product_exact_alias"
        )

    if is_archive:
        score += 25
        reasons.append(
            "archive_extension"
        )

    if expected_suffix in tokens:
        score += 20
        reasons.append(
            f"version_suffix_{expected_suffix}"
        )

    if any(
        alias in tokens
        for alias in CONUS_ALIASES
    ):
        score += 20
        reasons.append(
            "CONUS_token"
        )

    # Common LANDFIRE filename prefixes:
    # LC21...210 and LC22...220.
    if version_key == "LF2019L":
        if (
            "LC21" in tokens
            or "LF2019" in compact
            or "2019L" in compact
        ):
            score += 10
            reasons.append(
                "LF2019L_name_evidence"
            )

    if version_key == "LF2020":
        if (
            "LC22" in tokens
            or "LF2020" in compact
        ):
            score += 10
            reasons.append(
                "LF2020_name_evidence"
            )

    return score, ";".join(
        reasons
    )


candidate_rows = []

# LF2020 ScienceBase.
for _, r in sciencebase_files.iterrows():

    detected = r[
        "product_detected"
    ]

    if detected not in REQUIRED_PRODUCTS:
        continue

    score, reasons = candidate_score(
        "LF2020",
        detected,
        r["file_name"],
        bool(
            r[
                "is_archive"
            ]
        ),
    )

    candidate_rows.append({
        "version_key": "LF2020",
        "fire_year": 2022,
        "product": detected,
        "source_kind": "ScienceBase",
        "candidate_name": r[
            "file_name"
        ],
        "candidate_url": r[
            "file_url"
        ],
        "source_item_title": r[
            "item_title"
        ],
        "source_item_id": r[
            "item_id"
        ],
        "published_size_bytes": r[
            "file_size_bytes"
        ],
        "published_checksum_type": r[
            "checksum_type"
        ],
        "published_checksum_value": r[
            "checksum_value"
        ],
        "is_archive": bool(
            r[
                "is_archive"
            ]
        ),
        "candidate_score": score,
        "candidate_score_reasons": reasons,
    })


# LF2019L Help Desk.
for _, r in lf2019l_inventory.iterrows():

    if bool(
        r[
            "is_directory"
        ]
    ):
        continue

    detected = r[
        "product_detected"
    ]

    if detected not in REQUIRED_PRODUCTS:
        continue

    score, reasons = candidate_score(
        "LF2019L",
        detected,
        r["name"],
        bool(
            r[
                "is_archive"
            ]
        ),
    )

    candidate_rows.append({
        "version_key": "LF2019L",
        "fire_year": 2021,
        "product": detected,
        "source_kind": "HelpDesk_directory",
        "candidate_name": r[
            "name"
        ],
        "candidate_url": r[
            "url"
        ],
        "source_item_title": None,
        "source_item_id": None,
        "published_size_bytes": np.nan,
        "published_checksum_type": None,
        "published_checksum_value": None,
        "is_archive": bool(
            r[
                "is_archive"
            ]
        ),
        "candidate_score": score,
        "candidate_score_reasons": reasons,
    })


candidates = pd.DataFrame(
    candidate_rows
)

if len(
    candidates
):
    candidates = candidates.sort_values(
        [
            "version_key",
            "product",
            "candidate_score",
            "candidate_name",
        ],
        ascending=[
            True,
            True,
            False,
            True,
        ],
    ).reset_index(
        drop=True
    )

display(
    candidates
)

candidates.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_CANDIDATES_V2.csv",
    index=False,
)

## 9. Collect HEAD metadata for top-ranked candidates

In [ ]:
if candidates.empty:
    top_candidates = candidates.copy()

else:
    top_candidates = (
        candidates.groupby(
            [
                "version_key",
                "product",
            ],
            group_keys=False,
        )
        .head(3)
        .copy()
    )

head_rows = []

for _, r in top_candidates.iterrows():

    url = r[
        "candidate_url"
    ]

    meta = (
        head_metadata(
            url
        )
        if pd.notna(
            url
        )
        and str(
            url
        ).startswith(
            (
                "http://",
                "https://",
            )
        )
        else {
            "head_status": np.nan,
            "head_final_url": None,
            "head_content_length": np.nan,
            "head_content_type": None,
            "head_etag": None,
            "head_last_modified": None,
        }
    )

    head_rows.append({
        **r.to_dict(),
        **meta,
    })

top_candidate_head = pd.DataFrame(
    head_rows
)

display(
    top_candidate_head[
        [
            "version_key",
            "product",
            "candidate_name",
            "candidate_score",
            "published_size_bytes",
            "head_content_length",
            "published_checksum_type",
            "published_checksum_value",
            "head_status",
            "head_content_type",
        ]
    ]
    if len(
        top_candidate_head
    )
    else top_candidate_head
)

top_candidate_head.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_TOP_CANDIDATE_HEAD_V2.csv",
    index=False,
)

## 10. Conservative automatic recommendation

The notebook may recommend a candidate for each product, but it does not yet
authorize a download.

A recommendation requires:

- recognized required product;
- archive filename;
- uniquely highest candidate score within that `version × product`;
- no tied top candidate.

The resulting table is the artifact to inspect before enabling downloads in a
separate acquisition notebook.

In [ ]:
selection_rows = []

for version_key in [
    "LF2019L",
    "LF2020",
]:
    for product in REQUIRED_PRODUCTS:

        g = candidates[
            (
                candidates[
                    "version_key"
                ]
                == version_key
            )
            & (
                candidates[
                    "product"
                ]
                == product
            )
            & (
                candidates[
                    "is_archive"
                ]
                == True
            )
        ].copy()

        if g.empty:
            selection_rows.append({
                "version_key": version_key,
                "product": product,
                "selection_status": "NO_ARCHIVE_CANDIDATE",
                "recommended_name": None,
                "recommended_url": None,
                "candidate_score": np.nan,
                "tied_top_count": 0,
                "manual_review_required": True,
            })
            continue

        max_score = g[
            "candidate_score"
        ].max()

        top = g[
            g[
                "candidate_score"
            ]
            == max_score
        ].copy()

        if len(
            top
        ) != 1:

            selection_rows.append({
                "version_key": version_key,
                "product": product,
                "selection_status": "AMBIGUOUS_TOP_CANDIDATE",
                "recommended_name": None,
                "recommended_url": None,
                "candidate_score": max_score,
                "tied_top_count": len(top),
                "manual_review_required": True,
            })
            continue

        r = top.iloc[
            0
        ]

        selection_rows.append({
            "version_key": version_key,
            "product": product,
            "selection_status": "UNIQUE_TOP_CANDIDATE",
            "recommended_name": r[
                "candidate_name"
            ],
            "recommended_url": r[
                "candidate_url"
            ],
            "candidate_score": r[
                "candidate_score"
            ],
            "candidate_score_reasons": r[
                "candidate_score_reasons"
            ],
            "published_size_bytes": r[
                "published_size_bytes"
            ],
            "published_checksum_type": r[
                "published_checksum_type"
            ],
            "published_checksum_value": r[
                "published_checksum_value"
            ],
            "tied_top_count": 1,
            "manual_review_required": False,
        })


selection = pd.DataFrame(
    selection_rows
)

display(
    selection
)

selection.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_RECOMMENDED_SELECTION_V2.csv",
    index=False,
)

## 11. Acquisition readiness checks

In [ ]:
checks = []

for version_key in [
    "LF2019L",
    "LF2020",
]:
    s = selection[
        selection[
            "version_key"
        ]
        == version_key
    ]

    checks.append({
        "check": (
            f"{version_key}_five_required_products_present"
        ),
        "pass": (
            len(s) == 5
            and (
                s[
                    "selection_status"
                ]
                == "UNIQUE_TOP_CANDIDATE"
            ).all()
        ),
    })


checks.extend([
    {
        "check": "exactly_ten_version_product_contract_rows",
        "pass": len(
            acquisition_contract
        ) == 10,
    },
    {
        "check": "archive_download_disabled",
        "pass": (
            ALLOW_ARCHIVE_DOWNLOAD
            is False
        ),
    },
    {
        "check": "LF2020_DOI_resolved_to_ScienceBase",
        "pass": bool(
            lf2020_resolution.get(
                "sciencebase_item_id"
            )
        ),
    },
    {
        "check": "LF2020_descendant_hydration_complete",
        "pass": bool(
            len(descendant_summaries)
            == len(hydrated_descendants)
            and (
                len(sciencebase_hydration_log) == 0
                or not (
                    sciencebase_hydration_log[
                        "hydration_status"
                    ] == "failed"
                ).any()
            )
        ),
    },
    {
        "check": "no_candidate_download_performed",
        "pass": True,
    },
])

checks_df = pd.DataFrame(
    checks
)

display(
    checks_df
)

INVENTORY_READY = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "04D1F ARCHIVE INVENTORY:",
    (
        "PASS / READY FOR HUMAN REVIEW"
        if INVENTORY_READY
        else "REVIEW CANDIDATE INVENTORY"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_INVENTORY_CHECKS_V2.csv",
    index=False,
)

## 12. Explicit no-download guard

This cell exists so that accidentally changing a later notebook cell cannot
silently turn this inventory notebook into a large downloader.

In [ ]:
def download_archive(*args, **kwargs):
    if not ALLOW_ARCHIVE_DOWNLOAD:
        raise RuntimeError(
            "Archive download is disabled in 04D1F inventory mode. "
            "Review the candidate/source manifest first and use the "
            "next acquisition notebook for downloads."
        )

    raise RuntimeError(
        "04D1F has no archive-download implementation by design."
    )


try:
    download_archive(
        "safety-test"
    )

except RuntimeError as exc:
    assert (
        "disabled"
        in str(
            exc
        ).lower()
    )

print(
    "NO-DOWNLOAD GUARD: PASS"
)

## 13. Provenance manifest

In [ ]:
manifest_rows = []

selection_lookup = {
    (
        r[
            "version_key"
        ],
        r[
            "product"
        ],
    ): r
    for _, r in selection.iterrows()
}

for _, c in acquisition_contract.iterrows():

    key = (
        c[
            "version_key"
        ],
        c[
            "product"
        ],
    )

    s = selection_lookup.get(
        key
    )

    manifest_rows.append({
        "version_key": c[
            "version_key"
        ],
        "fire_year": c[
            "fire_year"
        ],
        "product": c[
            "product"
        ],
        "expected_capable_year": SOURCE_PLAN[
            c[
                "version_key"
            ]
        ][
            "expected_capable_year"
        ],
        "expected_version_suffix": c[
            "expected_version_suffix"
        ],
        "official_source_url": c[
            "official_source_url"
        ],
        "source_kind": c[
            "source_kind"
        ],
        "candidate_status": (
            None
            if s is None
            else s[
                "selection_status"
            ]
        ),
        "candidate_filename": (
            None
            if s is None
            else s[
                "recommended_name"
            ]
        ),
        "candidate_download_url": (
            None
            if s is None
            else s[
                "recommended_url"
            ]
        ),
        "published_size_bytes": (
            np.nan
            if s is None
            else s.get(
                "published_size_bytes",
                np.nan,
            )
        ),
        "published_checksum_type": (
            None
            if s is None
            else s.get(
                "published_checksum_type"
            )
        ),
        "published_checksum_value": (
            None
            if s is None
            else s.get(
                "published_checksum_value"
            )
        ),
        "download_status": "NOT_DOWNLOADED",
        "local_archive_path": None,
        "local_sha256": None,
        "zip_integrity_pass": None,
        "geotiff_found": None,
        "xml_metadata_found": None,
        "csv_attribute_table_found": None,
        "raster_crs": None,
        "raster_pixel_size_m": None,
        "raster_dtype": None,
        "raster_nodata": None,
        "raw_domain_pass": None,
        "science_provenance_status": (
            "inventory_candidate_only"
        ),
    })


provenance_manifest = pd.DataFrame(
    manifest_rows
)

display(
    provenance_manifest
)

provenance_manifest.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_PROVENANCE_MANIFEST_V2.csv",
    index=False,
)

## 14. Inventory report

In [ ]:
def human_bytes(
    value,
):
    if pd.isna(
        value
    ):
        return None

    x = float(
        value
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:
        if abs(
            x
        ) < 1024:
            return (
                f"{x:.2f} {unit}"
            )

        x /= 1024.0

    return (
        f"{x:.2f} PiB"
    )


selection_report = (
    selection.copy()
)

if "published_size_bytes" in selection_report.columns:
    selection_report[
        "published_size_human"
    ] = (
        selection_report[
            "published_size_bytes"
        ].apply(
            human_bytes
        )
    )


def tomd(df):
    try:
        return df.to_markdown(
            index=False
        )

    except Exception:
        return df.to_string(
            index=False
        )


report = f"""# 04D1F Archived LANDFIRE Inventory & Provenance

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Frozen archived-vintage contract

{tomd(acquisition_contract)}

## LF2020 DOI resolution

ScienceBase item ID: `{lf2020_resolution['sciencebase_item_id']}`

Resolved URL: `{lf2020_resolution['final_url']}`

ScienceBase root title: `{root_item.get('title')}`

Descendant search summaries discovered: {len(descendant_summaries):,}

Full descendant items hydrated: {len(hydrated_descendants):,}

Hydration failures: {int((sciencebase_hydration_log["hydration_status"] == "failed").sum()) if len(sciencebase_hydration_log) else 0:,}

Attached files inspected: {len(sciencebase_files):,}

## LF2019 Limited Help Desk inventory

Directory entries inspected: {len(lf2019l_inventory):,}

## Recommended candidates

{tomd(selection_report)}

## Inventory checks

{tomd(checks_df)}

## Decision

`{"PASS / READY FOR HUMAN REVIEW" if INVENTORY_READY else "REVIEW CANDIDATE INVENTORY"}`

## Important

No archive file was downloaded by this notebook.

The next notebook may download only the reviewed/accepted five products for
each archived vintage, compute local checksums, validate ZIP contents and raw
GeoTIFF semantics, and then run the frozen V5 blockwise extraction architecture.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_INVENTORY_REPORT_V2.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 15. Output inventory

In [ ]:
print("=" * 100)
print("04D1F V2 OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)